# Listings — Diagnóstico de Limpieza y Tratamiento ETL

Pipeline bronze → silver sobre el dataset de listings de Airbnb (CDMX).

El notebook se ejecuta de arriba a abajo. Cada etapa registra filas/columnas y acumula
métricas en `METRICS`, que al final se consolida en la tabla de diagnóstico exportada.

## 0. Parámetros

Rutas y umbrales centralizados: nada de valores quemados dentro de las celdas de transformación.

In [ ]:
from pathlib import Path

# Rutas del pipeline. Ajustar solo aquí si cambia la estructura de carpetas.
BRONZE_PATH  = Path("../data/bronze/listings.csv")
SILVER_PATH  = Path("../data/silver/listings.csv")
FIGURES_DIR  = Path("../reports/figures")
DIAG_PATH    = Path("../reports/diagnostico_limpieza.csv")

# Clave de negocio del listing. El _id de Mongo es único por documento y no sirve para deduplicar.
BUSINESS_KEY = "id"

# Umbrales de descarte de outliers. Revisar los boxplots de la sección 2 antes de fijarlos.
PRICE_MIN_MXN   = 1        # descarta precios en 0 o negativos: no representan una tarifa real
PRICE_MAX_MXN   = 100_000  # tope por noche; ajustar tras inspeccionar el percentil 99.9
CAPACITY_MAX    = 15       # tope para bathrooms / bedrooms / beds
TOP_N_AMENITIES = 10

# La moneda del dataset es MXN (scrape de Ciudad de México). Se asume formato numérico
# en_US en el CSV crudo ("$1,234.56"); la celda 2.4 valida ese supuesto antes de convertir.
CURRENCY = "MXN"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DIAG_PATH.parent.mkdir(parents=True, exist_ok=True)
SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
import ast
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
matplotlib.rcParams["figure.dpi"] = 110

# Acumulador de métricas del diagnóstico. Cada etapa escribe aquí en vez de solo imprimir,
# para que la tabla final se derive de la ejecución real y no de valores transcritos a mano.
METRICS = []


def log_step(df, etapa, detalle=""):
    """Registra el estado del DataFrame tras una etapa del pipeline."""
    METRICS.append({
        "etapa": etapa,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "detalle": detalle,
    })
    print(f"[{etapa}] filas={df.shape[0]:,} columnas={df.shape[1]} {detalle}")


def save_fig(fig, nombre, mostrar=True):
    """Guarda la figura como entregable y la muestra en el notebook.

    display() se llama antes de plt.close() porque el backend inline solo renderiza
    las figuras que siguen abiertas al terminar la celda; cerrar sin mostrar las
    dejaba únicamente en disco. Se cierra después para no duplicar la imagen
    ni acumular figuras en memoria.
    """
    ruta = FIGURES_DIR / f"{nombre}.png"
    fig.savefig(ruta, bbox_inches="tight")
    print(f"figura guardada: {ruta}")
    if mostrar:
        display(fig)
    plt.close(fig)
    return ruta


## 1. Ingesta — capa bronze

Lectura del archivo crudo sin conversiones implícitas: `price`, las tasas con `%` y las
columnas de lista se dejan como texto para poder diagnosticar su formato original antes
de transformarlas.

In [ ]:
df_raw = pd.read_csv(BRONZE_PATH, sep=",", encoding="utf-8", low_memory=False)
log_step(df_raw, "01_ingesta_bronze", f"fuente={BRONZE_PATH}")
df_raw.head()

In [ ]:
resumen_tipos = (
    df_raw.dtypes.value_counts()
    .rename_axis("dtype")
    .reset_index(name="columnas")
)
display(resumen_tipos)
df_raw.info(verbose=False)

## 2. Diagnóstico del dato crudo

### 2.1 Registros duplicados

El dataset contiene **varios scrapes**, por lo que un mismo listing aparece más de una vez
con distinta `last_scraped`. Deduplicar sobre las 77 columnas no elimina nada porque `_id`
es un ObjectId único por documento: la clave de negocio real es `id`.

In [ ]:
dup_exactos = df_raw.duplicated().sum()
dup_sin_id  = df_raw.drop(columns=["_id"], errors="ignore").duplicated().sum()
dup_negocio = df_raw.duplicated(subset=[BUSINESS_KEY]).sum()

print(f"Duplicados exactos (77 columnas, incluye _id): {dup_exactos:,}")
print(f"Duplicados ignorando _id:                      {dup_sin_id:,}")
print(f"Duplicados por clave de negocio '{BUSINESS_KEY}':  {dup_negocio:,}")
print(f"Listings únicos: {df_raw[BUSINESS_KEY].nunique():,} de {len(df_raw):,} filas")

In [ ]:
# Distribución de scrapes: confirma cuántas cargas conviven en el archivo bronze.
print("scrape_id distintos:", df_raw["scrape_id"].nunique())
display(df_raw["scrape_id"].value_counts().rename_axis("scrape_id").reset_index(name="filas"))
display(df_raw["last_scraped"].value_counts().rename_axis("last_scraped").reset_index(name="filas"))

# Cuántas veces se repite cada listing entre scrapes.
repeticiones = df_raw[BUSINESS_KEY].value_counts().value_counts().sort_index()
display(repeticiones.rename_axis("apariciones_del_listing").reset_index(name="cantidad_de_listings"))

In [ ]:
# Muestra de un listing duplicado, para verificar que las filas difieren solo por fecha de scrape
# y campos de estado (disponibilidad, reviews) y no por atributos estructurales.
ids_dup = df_raw.loc[df_raw.duplicated(subset=[BUSINESS_KEY], keep=False), BUSINESS_KEY].unique()
if len(ids_dup) > 0:
    muestra = df_raw[df_raw[BUSINESS_KEY] == ids_dup[0]]
    cols_muestra = [c for c in [BUSINESS_KEY, "scrape_id", "last_scraped", "calendar_last_scraped",
                                "price", "availability_365", "number_of_reviews"]
                    if c in df_raw.columns]
    display(muestra[cols_muestra])
else:
    print("No hay listings repetidos entre scrapes.")

### 2.2 Valores faltantes

In [ ]:
nulos = df_raw.isnull().sum()
por_nulos = (df_raw.isnull().mean() * 100).round(2)
tabla_nulos = (
    pd.concat([nulos, por_nulos], axis=1, keys=["nulos", "pct"])
    .query("nulos > 0")
    .sort_values("nulos", ascending=False)
)
print(f"Columnas con al menos un nulo: {len(tabla_nulos)} de {df_raw.shape[1]}")
display(tabla_nulos)

In [ ]:
if not tabla_nulos.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(tabla_nulos.index, tabla_nulos["pct"], color="steelblue")
    ax.set_title("Porcentaje de valores nulos por columna — dato crudo")
    ax.set_xlabel("Columnas")
    ax.set_ylabel("% de nulos")
    ax.tick_params(axis="x", rotation=90)
    ax.axhline(45, color="firebrick", linestyle="--", linewidth=1, label="umbral 45%")
    ax.legend()
    save_fig(fig, "01_nulos_pct_por_columna")

In [ ]:
# Mapa de calor sobre las filas que tienen al menos un nulo: revela si los faltantes
# aparecen en bloque (patrón estructural) o dispersos (error de captura).
cols_con_nulos = tabla_nulos.index.tolist()
if cols_con_nulos:
    df_filas_nulas = df_raw.loc[df_raw[cols_con_nulos].isnull().any(axis=1), cols_con_nulos]
    fig, ax = plt.subplots(figsize=(14, 7))
    sns.heatmap(df_filas_nulas.isnull(), cbar=False, cmap="viridis", ax=ax)
    ax.set_title("Patrón de valores nulos")
    ax.set_xlabel("Columnas")
    ax.set_ylabel("Filas con al menos un nulo")
    save_fig(fig, "02_heatmap_nulos")

In [ ]:
# Verificación del patrón: los nulos de reviews deberían concentrarse en listings sin reviews.
sin_reviews = df_raw["number_of_reviews"] == 0
cols_review = [c for c in df_raw.columns if c.startswith("review_scores_")] + \
              ["first_review", "last_review", "reviews_per_month"]
cols_review = [c for c in cols_review if c in df_raw.columns]

comparacion = pd.DataFrame({
    "pct_nulos_listings_sin_reviews": df_raw.loc[sin_reviews, cols_review].isnull().mean().mul(100).round(2),
    "pct_nulos_listings_con_reviews": df_raw.loc[~sin_reviews, cols_review].isnull().mean().mul(100).round(2),
})
print(f"Listings sin ninguna review: {sin_reviews.sum():,} ({sin_reviews.mean()*100:.1f}%)")
display(comparacion)

### 2.3 Inconsistencias tipográficas

`neighbourhood` es texto libre capturado por el anfitrión. Se agrupan los valores por su
forma normalizada (minúsculas, sin acentos, sin espacios redundantes) para cuantificar
cuántas variantes de escritura corresponden a la misma zona.

In [ ]:
def normalize_text(text):
    """Forma canónica para comparar cadenas escritas por el usuario."""
    if pd.isnull(text):
        return text
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", errors="ignore").decode("utf-8")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


col_texto = "neighbourhood"
serie = df_raw[col_texto].dropna()
variantes = (
    pd.DataFrame({"crudo": serie, "normalizado": serie.map(normalize_text)})
    .drop_duplicates()
    .groupby("normalizado")["crudo"]
    .agg(list)
)
colisiones = variantes[variantes.map(len) > 1]

print(f"Valores únicos crudos:       {serie.nunique():,}")
print(f"Valores únicos normalizados: {serie.map(normalize_text).nunique():,}")
print(f"Grupos con más de una grafía: {len(colisiones)}")
for norm, formas in colisiones.head(15).items():
    print(f"  {norm!r} <- {formas}")

### 2.4 Formato del campo `price`

Antes de convertir hay que confirmar el formato exacto del texto crudo. La versión anterior
del notebook eliminaba el punto junto con el `$` y la coma, lo que borraba el separador
decimal y multiplicaba todos los precios por 100.

In [ ]:
muestra_price = df_raw["price"].dropna().astype(str)
print("Muestra de valores crudos:")
print(muestra_price.head(15).tolist())

# Los conteos se calculan fuera de los f-strings: hasta Python 3.11 un backslash
# dentro de las llaves de un f-string es SyntaxError, y el kernel del proyecto es 3.11.
con_simbolo  = muestra_price.str.contains("$", regex=False).sum()
con_coma     = muestra_price.str.contains(",", regex=False).sum()
con_punto    = muestra_price.str.contains(".", regex=False).sum()
con_centavos = muestra_price.str.endswith(".00").sum()

print()
print(f"Contienen '$':           {con_simbolo:,}")
print(f"Contienen ',' (miles):   {con_coma:,}")
print(f"Contienen '.' (decimal): {con_punto:,}")
print(f"Terminan en '.00':       {con_centavos:,}")

# Un valor con más de un punto indicaría formato es_MX (punto como separador de miles)
# y obligaría a invertir la lógica de limpieza.
n_puntos = muestra_price.str.count(r"\.")
multi_punto = muestra_price[n_puntos > 1]
print(f"Valores con más de un punto: {len(multi_punto):,}")
if len(multi_punto):
    print(multi_punto.head(10).tolist())


### 2.5 Estadísticas descriptivas y distribuciones

In [ ]:
df_num = df_raw.select_dtypes(include=["number"])
display(df_num.describe().T.round(2))

In [ ]:
cols_hist = [c for c in df_num.columns if df_num[c].nunique() > 1]
n_cols = 4
n_rows = int(np.ceil(len(cols_hist) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.2, n_rows * 3.0))
for ax, col in zip(axes.ravel(), cols_hist):
    sns.histplot(df_num[col].dropna(), bins=30, color="royalblue", ax=ax)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
    ax.set_ylabel("")
for ax in axes.ravel()[len(cols_hist):]:
    ax.axis("off")
fig.suptitle("Distribución de variables numéricas — dato crudo", y=1.001)
fig.tight_layout()
save_fig(fig, "03_histogramas_numericas")

In [ ]:
# Boxplots de las variables sobre las que se toma una decisión de outliers.
# El loop original generaba una figura por columna (~50 archivos) sin valor para el reporte.
cols_box = [c for c in ["accommodates", "bathrooms", "bedrooms", "beds",
                        "minimum_nights", "maximum_nights", "host_listings_count",
                        "number_of_reviews", "availability_365"]
            if c in df_raw.columns]

fig, axes = plt.subplots(len(cols_box), 1, figsize=(10, 1.8 * len(cols_box)))
for ax, col in zip(np.atleast_1d(axes), cols_box):
    sns.boxplot(x=df_raw[col], color="skyblue", ax=ax)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
fig.suptitle("Valores atípicos — variables numéricas clave", y=1.001)
fig.tight_layout()
save_fig(fig, "04_boxplots_numericas_crudo")

## 3. Transformaciones

Todas las etapas encadenan sobre `df_silver`. Cada celda parte del resultado de la anterior:
no se vuelve a referenciar `df_raw` después de la copia inicial.

### 3.1 Deduplicación por clave de negocio

Se conserva la observación más reciente de cada listing. El criterio de recencia es
`last_scraped` y, como desempate, `calendar_last_scraped`: entre dos capturas del mismo
anuncio la última refleja el estado vigente de precio, disponibilidad y reviews.

In [ ]:
df_silver = df_raw.copy()
filas_ini = len(df_silver)

cols_orden = [c for c in ["last_scraped", "calendar_last_scraped", "scrape_id"] if c in df_silver.columns]
for c in ["last_scraped", "calendar_last_scraped"]:
    if c in df_silver.columns:
        df_silver[c] = pd.to_datetime(df_silver[c], errors="coerce")

df_silver = (
    df_silver.sort_values([BUSINESS_KEY] + cols_orden)
             .drop_duplicates(subset=[BUSINESS_KEY], keep="last")
             .reset_index(drop=True)
)
eliminadas = filas_ini - len(df_silver)
log_step(df_silver, "02_dedup_por_listing", f"eliminadas={eliminadas:,} (scrapes anteriores del mismo listing)")

assert df_silver[BUSINESS_KEY].is_unique, "La clave de negocio sigue duplicada tras la deduplicación"

### 3.2 Corrección de `price`

Se eliminan `$` y la coma de miles, y **se conserva el punto decimal**. El resultado se
redondea a 2 decimales, la escala real de una tarifa monetaria. Al materializar esta tabla
en Delta el campo debe declararse `DECIMAL(12,2)`: `float64` no es un tipo válido para
importes porque acumula error en agregaciones.

In [ ]:
serie_price = df_silver["price"].astype("string").str.strip()
nulos_antes = serie_price.isna().sum()

serie_price = (serie_price
               .str.replace("$", "", regex=False)
               .str.replace(",", "", regex=False))

# Si algún valor conserva más de un punto, el supuesto de formato en_US es falso
# y la limpieza anterior estaría corrompiendo el importe.
sospechosos = serie_price.dropna()[serie_price.dropna().str.count(r"\.") > 1]
assert sospechosos.empty, f"Formato numérico inesperado en price: {sospechosos.head().tolist()}"

df_silver["price"] = pd.to_numeric(serie_price, errors="coerce").round(2)
nulos_despues = df_silver["price"].isna().sum()

print(f"Nulos antes: {nulos_antes:,} | después: {nulos_despues:,} | no convertibles: {nulos_despues - nulos_antes:,}")
display(df_silver["price"].describe().round(2))

In [ ]:
# Validación cruzada del importe: el ingreso estimado del último año debe aproximar
# price * noches ocupadas. Con el bug anterior (punto decimal eliminado) este ratio
# daba ~100; corregido debe rondar 1.
if {"estimated_revenue_l365d", "estimated_occupancy_l365d"}.issubset(df_silver.columns):
    val = df_silver[(df_silver["estimated_occupancy_l365d"] > 0) & (df_silver["price"] > 0)].copy()
    val["price_implicito"] = val["estimated_revenue_l365d"] / val["estimated_occupancy_l365d"]
    val["ratio"] = (val["price_implicito"] / val["price"]).round(3)
    print(f"Registros validables: {len(val):,}")
    print(f"Ratio mediano price_implicito/price: {val['ratio'].median():.3f}  (esperado ~1.0)")
    display(val["ratio"].describe().round(3))

### 3.3 Tasas con símbolo `%`

In [ ]:
cols_rate = [c for c in ["host_response_rate", "host_acceptance_rate"] if c in df_silver.columns]
for col in cols_rate:
    df_silver[col] = pd.to_numeric(
        df_silver[col].astype("string").str.replace("%", "", regex=False),
        errors="coerce",
    )
display(df_silver[cols_rate].describe().round(2))

### 3.4 Categorización de `host_response_time`

Se agrupan las cuatro categorías originales en tres niveles. Se conserva la columna original
hasta el subset final para poder auditar el mapeo.

In [ ]:
response_time_map = {
    "within an hour": "Fast",
    "within a few hours": "Fast",
    "within a day": "Moderate",
    "a few days or more": "Slow",
}

no_mapeados = set(df_silver["host_response_time"].dropna().unique()) - set(response_time_map)
assert not no_mapeados, f"Categorías sin mapeo en host_response_time: {no_mapeados}"

df_silver["host_response_category"] = df_silver["host_response_time"].map(response_time_map)
display(df_silver["host_response_category"].value_counts(dropna=False))

### 3.5 Binarias de `host_verifications`

Se parsea la lista y se generan indicadores con `explode` + `get_dummies`, alineados por
índice contra `df_silver`. El `concat` de la versión anterior se hacía contra `df` original,
lo que descartaba todas las transformaciones previas y reintroducía los duplicados.

In [ ]:
def parse_list(value):
    """Convierte la representación textual de lista a lista de Python; tolera nulos y basura."""
    if isinstance(value, list):
        return value
    if pd.isnull(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


verifs = df_silver["host_verifications"].apply(parse_list)

# parse_list devuelve [] ante un formato no reconocido. Sin esta verificación un cambio
# de formato en el origen produciría todas las binarias en cero sin ningún error.
vacias = verifs.map(len).eq(0).sum()
nulos_hv = df_silver["host_verifications"].isna().sum()
assert vacias <= nulos_hv, (
    f"{vacias - nulos_hv:,} filas no nulas de host_verifications no se pudieron parsear "
    "como lista literal de Python. Revisar el formato del campo en bronze."
)

metodos = sorted({m for lista in verifs for m in lista})
print("Métodos de verificación encontrados:", metodos)

dummies = (
    pd.get_dummies(verifs.explode().dropna())
    .groupby(level=0).max()
    .reindex(df_silver.index, fill_value=0)
    .astype("int8")
)
dummies.columns = [f"verif_{re.sub(r'[^a-z0-9]+', '_', str(c).lower()).strip('_')}" for c in dummies.columns]

df_silver = pd.concat([df_silver, dummies], axis=1)
log_step(df_silver, "03_binarias_verificaciones", f"nuevas={list(dummies.columns)}")

### 3.6 Amenities: top-10 e indicador de equipamiento

El parseo se hace con `ast.literal_eval` sobre la columna cruda. La versión anterior aplicaba
`astype(str)` sobre listas ya parseadas y luego partía por coma, con lo que los tokens
quedaban con corchetes y comillas y los nombres de columna generados eran inválidos.

In [ ]:
amenities = df_silver["amenities"].apply(parse_list).apply(
    lambda lista: [str(a).strip().lower() for a in lista if str(a).strip()]
)

vacias_am = amenities.map(len).eq(0).sum()
nulos_am = df_silver["amenities"].isna().sum()
assert vacias_am <= nulos_am, (
    f"{vacias_am - nulos_am:,} filas no nulas de amenities no se pudieron parsear como lista. "
    "Revisar el formato del campo en bronze antes de generar las binarias."
)

frecuencias = Counter(a for lista in amenities for a in lista)
top_amenities = [a for a, _ in frecuencias.most_common(TOP_N_AMENITIES)]

print(f"Amenities únicos: {len(frecuencias):,}")
display(pd.DataFrame(frecuencias.most_common(TOP_N_AMENITIES), columns=["amenity", "frecuencia"]))

In [ ]:
sets_amenities = amenities.map(set)
for amenity in top_amenities:
    col = "amen_" + re.sub(r"[^a-z0-9]+", "_", amenity).strip("_")
    df_silver[col] = sets_amenities.map(lambda s, a=amenity: int(a in s)).astype("int8")

# Conteo sobre la lista parseada: partir el string por coma inflaba el valor
# en amenities cuyo nombre contiene comas.
df_silver["amenities_count"] = amenities.map(len).astype("int16")

log_step(df_silver, "04_binarias_amenities", f"top_{TOP_N_AMENITIES}={top_amenities}")
display(df_silver["amenities_count"].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
top_df = pd.DataFrame(frecuencias.most_common(TOP_N_AMENITIES), columns=["amenity", "frecuencia"])
ax.barh(top_df["amenity"][::-1], top_df["frecuencia"][::-1], color="steelblue")
ax.set_title(f"Top {TOP_N_AMENITIES} amenities más frecuentes")
ax.set_xlabel("Listings que la ofrecen")
save_fig(fig, "05_top_amenities")

### 3.7 Normalización de `bathrooms` y de texto

`bathrooms` se redondea hacia arriba: un registro con 5.5 baños corresponde a cinco completos
más un medio baño, que sigue siendo una unidad sanitaria utilizable. `bathrooms_text` se
mantiene hasta el subset por si se requiere auditar el redondeo.

`neighbourhood` se normaliza y **se conserva**: eliminarla después de normalizarla, como en la
versión anterior, deja el análisis geográfico sin llave de agrupación.

In [ ]:
df_silver["bathrooms"] = np.ceil(df_silver["bathrooms"]).astype("Int64")
df_silver["neighbourhood"] = df_silver["neighbourhood"].map(normalize_text)

print("Baños (valores únicos):", sorted(df_silver["bathrooms"].dropna().unique().tolist()))
print("Barrios únicos tras normalizar:", df_silver["neighbourhood"].nunique())

### 3.8 Tipado de banderas booleanas

El origen es Mongo, por lo que estos campos pueden llegar como `True/False` o como los
`t/f` del CSV nativo de InsideAirbnb. La conversión cubre ambos formatos.

In [ ]:
cols_bool = [c for c in ["host_is_superhost", "host_has_profile_pic", "host_identity_verified",
                         "has_availability", "instant_bookable"]
             if c in df_silver.columns]

mapa_bool = {"t": True, "f": False, "true": True, "false": False, "1": True, "0": False}
for col in cols_bool:
    df_silver[col] = (df_silver[col].astype("string").str.strip().str.lower()
                      .map(mapa_bool).astype("boolean"))

display(df_silver[cols_bool].apply(lambda s: s.value_counts(dropna=False)).T)

### 3.9 Imputación

Solo se imputa donde el faltante tiene un valor conocido por regla de negocio:
`reviews_per_month` es 0 para un listing sin ninguna review, no un dato desconocido.

Las métricas `review_scores_*`, `first_review` y `last_review` **no se imputan**: el nulo
significa "sin reviews" y rellenarlo con la media inventaría una calificación que el
anuncio nunca recibió y sesgaría cualquier modelo posterior. Igual criterio para
`host_response_rate` y `host_acceptance_rate`, donde el nulo indica que el anfitrión aún
no tiene historial de respuesta.

In [ ]:
sin_reviews = df_silver["number_of_reviews"] == 0
imputados = (df_silver.loc[sin_reviews, "reviews_per_month"].isna()).sum()
df_silver.loc[sin_reviews, "reviews_per_month"] = df_silver.loc[sin_reviews, "reviews_per_month"].fillna(0)

residuo = df_silver["reviews_per_month"].isna().sum()
log_step(df_silver, "05_imputacion", f"reviews_per_month imputados={imputados:,}, nulos restantes={residuo:,}")

### 3.10 Tratamiento de outliers

Se descartan solo los valores que no admiten lectura de negocio. Los precios altos de
propiedades de lujo, los anfitriones con cientos de anuncios y las estancias mínimas largas
se conservan: son variación real del mercado, no error de captura.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.5))
sns.boxplot(x=df_silver["price"], color="skyblue", ax=ax)
ax.set_title(f"price ({CURRENCY}) — tras corregir el separador decimal, antes de recortar")
save_fig(fig, "06_boxplot_price_pre_recorte")

display(df_silver["price"].quantile([0.5, 0.9, 0.99, 0.999, 1.0]).round(2))

In [ ]:
filas_antes = len(df_silver)

fuera_rango = (df_silver["price"] < PRICE_MIN_MXN) | (df_silver["price"] > PRICE_MAX_MXN)
print(f"Fuera de [{PRICE_MIN_MXN:,}, {PRICE_MAX_MXN:,}] {CURRENCY}: {fuera_rango.sum():,}")
df_silver = df_silver[~fuera_rango | df_silver["price"].isna()].copy()

cols_capacidad = [c for c in ["bathrooms", "bedrooms", "beds"] if c in df_silver.columns]
excede = (df_silver[cols_capacidad] > CAPACITY_MAX).any(axis=1)
print(f"Con {cols_capacidad} > {CAPACITY_MAX}: {excede.sum():,}")
df_silver = df_silver[~excede].copy().reset_index(drop=True)

log_step(df_silver, "06_outliers", f"eliminadas={filas_antes - len(df_silver):,}")

## 4. Subset curado

Se descartan las columnas sin valor analítico: identificadores de scrape, URLs, imágenes,
texto libre largo y las derivadas redundantes de `minimum_nights` / `maximum_nights`.
La selección se interseca con las columnas realmente presentes para que el notebook no
falle si el esquema del scrape cambia.

In [ ]:
COLUMNAS_SILVER = [
    # Identificación
    "id", "name", "host_id",
    # Anfitrión
    "host_since", "host_is_superhost", "host_identity_verified",
    "host_response_category", "host_response_rate", "host_acceptance_rate",
    "host_listings_count", "verif_email", "verif_phone", "verif_work_email",
    # Ubicación
    "neighbourhood", "neighbourhood_cleansed", "latitude", "longitude",
    # Propiedad
    "property_type", "room_type", "accommodates", "bathrooms", "bedrooms", "beds",
    "amenities_count",
    # Precio y estancia
    "price", "minimum_nights", "maximum_nights",
    # Disponibilidad
    "has_availability", "availability_30", "availability_90", "availability_365",
    "instant_bookable",
    # Demanda y reputación
    "number_of_reviews", "number_of_reviews_ltm", "reviews_per_month",
    "first_review", "last_review",
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication", "review_scores_location",
    "review_scores_value",
    "estimated_occupancy_l365d", "estimated_revenue_l365d",
    # Trazabilidad del scrape que originó la fila
    "last_scraped",
]
COLUMNAS_SILVER += sorted(c for c in df_silver.columns if c.startswith("amen_"))

presentes = [c for c in COLUMNAS_SILVER if c in df_silver.columns]
faltantes = [c for c in COLUMNAS_SILVER if c not in df_silver.columns]
descartadas = [c for c in df_silver.columns if c not in presentes]

if faltantes:
    print(f"ADVERTENCIA — columnas esperadas ausentes en el origen: {faltantes}")
print(f"Se conservan {len(presentes)} columnas y se descartan {len(descartadas)}:")
print(descartadas)

df_silver = df_silver[presentes].copy()
log_step(df_silver, "07_subset_curado", f"descartadas={len(descartadas)}")

## 5. Validaciones de salida

In [ ]:
assert df_silver["id"].is_unique, "Hay listings duplicados en la salida"
assert df_silver["price"].dropna().between(PRICE_MIN_MXN, PRICE_MAX_MXN).all(), "Precios fuera de rango"
assert not any(df_silver[c].dtype == "object" and df_silver[c].map(type).eq(list).any()
               for c in df_silver.columns), "Quedan columnas con listas de Python sin serializar"
assert df_silver.loc[df_silver["number_of_reviews"] == 0, "reviews_per_month"].fillna(-1).ne(-1).all(), \
    "Quedan nulos en reviews_per_month para listings sin reviews"

print("Validaciones OK")
display(df_silver.dtypes.rename("dtype").to_frame())
df_silver.head()

## 6. Gráficas derivadas de las transformaciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(x=df_silver["price"], color="skyblue", ax=axes[0])
axes[0].set_title(f"price ({CURRENCY}) — post limpieza")
sns.histplot(df_silver["price"].dropna(), bins=60, log_scale=(True, False),
             color="royalblue", ax=axes[1])
axes[1].set_title("Distribución de price (escala log)")
axes[1].set_xlabel(f"price ({CURRENCY})")
fig.tight_layout()
save_fig(fig, "07_price_post_limpieza")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
orden = df_silver["room_type"].value_counts().index
sns.countplot(data=df_silver, y="room_type", order=orden, color="steelblue", ax=axes[0])
axes[0].set_title("Listings por tipo de alojamiento")
axes[0].set_xlabel("Listings")
axes[0].set_ylabel("")

sns.boxplot(data=df_silver[df_silver["price"].notna()], x="price", y="room_type",
            order=orden, color="skyblue", ax=axes[1])
axes[1].set_xscale("log")
axes[1].set_title(f"price ({CURRENCY}) por tipo de alojamiento")
axes[1].set_ylabel("")
fig.tight_layout()
save_fig(fig, "08_price_por_room_type")

In [ ]:
if "host_response_category" in df_silver.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.countplot(data=df_silver, x="host_response_category",
                  order=["Fast", "Moderate", "Slow"], color="steelblue", ax=ax)
    ax.set_title("Tiempo de respuesta del anfitrión")
    ax.set_xlabel("")
    ax.set_ylabel("Listings")
    save_fig(fig, "09_host_response_category")

In [ ]:
cols_corr = [c for c in ["price", "accommodates", "bedrooms", "beds", "bathrooms",
                         "amenities_count", "number_of_reviews", "review_scores_rating",
                         "availability_365", "estimated_revenue_l365d"]
             if c in df_silver.columns]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df_silver[cols_corr].corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación entre variables del dataset limpio")
save_fig(fig, "10_correlacion")

## 7. Tabla consolidada de diagnóstico

Se deriva de `METRICS`, poblado durante la ejecución, más el resumen de nulos previo y
posterior a la limpieza.

In [ ]:
df_metricas = pd.DataFrame(METRICS)
df_metricas["filas_eliminadas"] = -df_metricas["filas"].diff().fillna(0).astype(int)
display(df_metricas)

In [ ]:
diagnostico = pd.DataFrame([
    {"problema": "Registros duplicados",
     "columnas_afectadas": BUSINESS_KEY,
     "magnitud": f"{dup_negocio:,} filas del mismo listing en distintos scrapes",
     "tecnica": "Eliminación",
     "justificacion": "El archivo bronze acumula varios scrapes; se conserva la captura más reciente por listing."},
    {"problema": "Separador decimal eliminado en price",
     "columnas_afectadas": "price",
     "magnitud": "100% de los registros con precio",
     "tecnica": "Transformación de tipo",
     "justificacion": "Se retiran '$' y coma de miles conservando el punto decimal; validado contra estimated_revenue/estimated_occupancy."},
    {"problema": "Símbolo '%' en variables de tasa",
     "columnas_afectadas": ", ".join(cols_rate),
     "magnitud": "Todas las filas no nulas",
     "tecnica": "Transformación de tipo",
     "justificacion": "Sin la conversión a numérico no admiten agregaciones ni análisis estadístico."},
    {"problema": "Columnas de lista en texto",
     "columnas_afectadas": "amenities, host_verifications",
     "magnitud": f"{len(frecuencias):,} amenities únicos, {len(metodos)} métodos de verificación",
     "tecnica": "Transformación / codificación binaria",
     "justificacion": f"Se generan indicadores para el top {TOP_N_AMENITIES} y un conteo total de servicios."},
    {"problema": "Inconsistencias tipográficas",
     "columnas_afectadas": "neighbourhood",
     "magnitud": f"{len(colisiones)} grupos con más de una grafía",
     "tecnica": "Normalización de texto",
     "justificacion": "Minúsculas, sin acentos y sin espacios redundantes para unificar la llave geográfica."},
    {"problema": "Valores faltantes estructurales",
     "columnas_afectadas": "review_scores_*, first_review, last_review",
     "magnitud": f"{sin_reviews.sum():,} listings sin ninguna review",
     "tecnica": "Conservación sin imputar",
     "justificacion": "El nulo significa 'sin reviews'; imputar la media inventaría calificaciones inexistentes."},
    {"problema": "Valores faltantes con valor conocido",
     "columnas_afectadas": "reviews_per_month",
     "magnitud": f"{imputados:,} registros",
     "tecnica": "Imputación con 0",
     "justificacion": "Un listing sin reviews tiene una tasa mensual de reviews igual a cero, no desconocida."},
    {"problema": "Columnas con alta proporción de nulos y sin valor analítico",
     "columnas_afectadas": "host_about, neighborhood_overview, host_neighbourhood, license",
     "magnitud": "Superan el 45% de nulos",
     "tecnica": "Eliminación",
     "justificacion": "Texto libre descriptivo; host_id conserva la capacidad de agrupar por anfitrión."},
    {"problema": "Valores atípicos no verosímiles",
     "columnas_afectadas": "price, bathrooms, bedrooms, beds",
     "magnitud": f"price fuera de [{PRICE_MIN_MXN:,}, {PRICE_MAX_MXN:,}] o capacidad > {CAPACITY_MAX}",
     "tecnica": "Eliminación",
     "justificacion": "Se descartan solo los imposibles de negocio; lujo, multi-host y estancias largas se conservan."},
])

diagnostico.to_csv(DIAG_PATH, index=False, encoding="utf-8")
print(f"Diagnóstico exportado: {DIAG_PATH.resolve()}")
display(diagnostico)

## 8. Persistencia — capa silver

In [ ]:
df_silver.to_csv(SILVER_PATH, index=False, encoding="utf-8")

verif = pd.read_csv(SILVER_PATH, low_memory=False)
assert verif.shape == df_silver.shape, f"Relectura inconsistente: {verif.shape} vs {df_silver.shape}"

log_step(df_silver, "08_guardado_silver", f"destino={SILVER_PATH}")
print(f"Archivo escrito: {SILVER_PATH.resolve()}")
print(f"Reducción total: {len(df_raw):,} -> {len(df_silver):,} filas | "
      f"{df_raw.shape[1]} -> {df_silver.shape[1]} columnas")
print(f"\nFiguras exportadas en {FIGURES_DIR.resolve()}:")
for f in sorted(FIGURES_DIR.glob("*.png")):
    print(" -", f.name)